# Bangalore Grocery Price Forecasting: Exploratory Data Analysis (EDA)

**Project:** AI-Based Grocery Price Prediction for Bangalore, Karnataka, India  
**Target Commodities:** Tomato, Onion, Potato  
**Data Source:** AGMARKNET Mandi Records (2021 – 2026)  
**Objective:** Analyze multi-year price dynamics, seasonality, volatility, and cross-commodity interactions to inform feature engineering and predictive modeling.

In [1]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set styling
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.sans-serif'] = 'Arial'
plt.rcParams['axes.edgecolor'] = '#cccccc'

# Load cleaned processed dataset
data_path = '../data/processed/bangalore_grocery_prices_processed.csv'
df = pd.read_csv(data_path)
df['date'] = pd.to_datetime(df['date'])
print(f"Loaded {len(df):,} records spanning {df['date'].min().strftime('%Y-%m-%d')} to {df['date'].max().strftime('%Y-%m-%d')}.")
df.head()

## 1. Summary Statistics by Commodity
Let's inspect the distribution, price range (₹/kg), and observation count for Tomato, Onion, and Potato in Bangalore.

In [2]:
stats_list = []
for comm in ['Tomato', 'Onion', 'Potato']:
    sub = df[df['commodity'] == comm]
    p = sub['modal_price_per_kg']
    stats_list.append({
        'Commodity': comm,
        'Records': len(sub),
        'Earliest Date': sub['date'].min().strftime('%Y-%m-%d'),
        'Latest Date': sub['date'].max().strftime('%Y-%m-%d'),
        'Mean Price (Rs./kg)': round(p.mean(), 2),
        'Std Dev (Rs./kg)': round(p.std(), 2),
        'Min Price (Rs./kg)': round(p.min(), 2),
        'Median (Rs./kg)': round(p.median(), 2),
        'Max Price (Rs./kg)': round(p.max(), 2),
        'IQR (Rs./kg)': round(p.quantile(0.75) - p.quantile(0.25), 2),
    })
stats_df = pd.DataFrame(stats_list)
display(stats_df)

## 2. Daily Price Trends (2021 – 2026)
The daily modal price reveals significant cyclical fluctuations, supply shocks, and seasonal peaks.

In [3]:
colors = {'Tomato': '#e74c3c', 'Onion': '#8e44ad', 'Potato': '#d35400'}
fig, ax = plt.subplots(figsize=(14, 5.5))
for comm in ['Tomato', 'Onion', 'Potato']:
    sub = df[df['commodity'] == comm].sort_values('date')
    ax.plot(sub['date'], sub['modal_price_per_kg'], label=comm, color=colors[comm], alpha=0.85, lw=1.3)
ax.set_title('Daily Modal Wholesale Price (Bangalore Mandi: 2021 - 2026)', fontsize=13, fontweight='bold')
ax.set_ylabel('Price (Rs. / kg)')
ax.set_xlabel('Date')
ax.legend(frameon=True, facecolor='white')
plt.tight_layout()
plt.show()

## 3. Short-Term (7-Day) vs Medium-Term (30-Day) Moving Averages
Smoothing high-frequency noise highlights underlying trend trajectories.

In [4]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 9), sharex=True)
for comm in ['Tomato', 'Onion', 'Potato']:
    sub = df[df['commodity'] == comm].sort_values('date').copy()
    sub['ma7'] = sub['modal_price_per_kg'].rolling(7, min_periods=2).mean()
    sub['ma30'] = sub['modal_price_per_kg'].rolling(30, min_periods=5).mean()
    ax1.plot(sub['date'], sub['ma7'], label=f"{comm} 7-Day MA", color=colors[comm], lw=1.5)
    ax2.plot(sub['date'], sub['ma30'], label=f"{comm} 30-Day MA", color=colors[comm], lw=1.8)

ax1.set_title('7-Day Moving Average (Short-Term Trend)', fontsize=12, fontweight='bold')
ax1.set_ylabel('Price (Rs. / kg)')
ax1.legend(frameon=True, facecolor='white')

ax2.set_title('30-Day Moving Average (Medium-Term Trend)', fontsize=12, fontweight='bold')
ax2.set_ylabel('Price (Rs. / kg)')
ax2.set_xlabel('Date')
ax2.legend(frameon=True, facecolor='white')
plt.tight_layout()
plt.show()

## 4. Price Distribution and Volatility
Examining the skewness, spread, and rolling 30-day standard deviation across commodities.

In [5]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4.8))
for comm in ['Tomato', 'Onion', 'Potato']:
    sub = df[df['commodity'] == comm]
    sns.kdeplot(sub['modal_price_per_kg'], ax=ax1, label=comm, color=colors[comm], fill=True, alpha=0.3, lw=1.8)
ax1.set_title('Price Density Distributions (KDE)', fontsize=12, fontweight='bold')
ax1.set_xlabel('Modal Price (Rs. / kg)')
ax1.set_ylabel('Density')
ax1.legend(frameon=True, facecolor='white')

sns.boxplot(data=df, x='commodity', y='modal_price_per_kg', ax=ax2, hue='commodity', palette=colors, width=0.5, legend=False)
ax2.set_title('Price Dispersion & Spikes', fontsize=12, fontweight='bold')
ax2.set_xlabel('Commodity')
ax2.set_ylabel('Modal Price (Rs. / kg)')
plt.tight_layout()
plt.show()

## 5. Intra-Year Seasonality and Cross-Commodity Correlation
Are commodity prices seasonal by calendar month? How correlated are Tomato, Onion, and Potato wholesale prices?

In [6]:
df['month_num'] = df['date'].dt.month
monthly_seasonality = df.groupby(['month_num', 'commodity'])['modal_price_per_kg'].mean().unstack()
month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
for comm in ['Tomato', 'Onion', 'Potato']:
    if comm in monthly_seasonality.columns:
        ax1.plot(range(1, 13), monthly_seasonality[comm], marker='o', label=comm, color=colors[comm], lw=2.0)
ax1.set_title('Intra-Year Monthly Seasonal Pattern', fontsize=12, fontweight='bold')
ax1.set_xlabel('Calendar Month')
ax1.set_ylabel('Average Price (Rs. / kg)')
ax1.set_xticks(range(1, 13))
ax1.set_xticklabels(month_names)
ax1.legend(frameon=True, facecolor='white')

# Correlation Matrix
price_pivot = df.pivot(index='date', columns='commodity', values='modal_price_per_kg').dropna()
corr = price_pivot.corr()
sns.heatmap(corr, annot=True, fmt='.3f', cmap='coolwarm', vmin=-1, vmax=1, ax=ax2, cbar_kws={'label': 'Pearson r'})
ax2.set_title('Cross-Commodity Correlation Matrix', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Key Findings for Modeling
1. **Tomato Dynamics:** Highest volatility (std = Rs.16.8/kg) with rapid supply surges and sharp price spikes (up to Rs.110/kg), primarily in mid-monsoon months (June–August).
2. **Onion Dynamics:** Moderate volatility with pronounced post-monsoon / festival season peaks (September–November) reaching Rs.76/kg.
3. **Potato Dynamics:** Relatively stable baselines (IQR = Rs.8/kg) with structural shift in late 2024.
4. **Feature Implications:** Lag 1, rolling 7-day mean/std, price spread, and cyclical month encodings are key drivers for next-day forecasting.